# The Observables

A trajectory is just data until you ask it questions.
Here are the ones pyCoop knows how to ask.

## Agreement

For player $i$, agreement is the absolute value of the local mean action,

$$
a_i = |\bar\sigma_i| \in [0, 1],
$$

computed by [`agreement()`](#pycoop.observables.agreement).
Note what the absolute value hides: a crowd torn between extremes and a crowd of fence-sitters score alike.
The alignment asymmetry below repairs that.

## Friends, enemies, and fraughtness

Fix a state with no player exactly at zero, and split player $i$'s neighbors by the sign of their *action*: opinions play no role here.
Friends share $i$'s sign, enemies oppose it, and each camp gets a weight,

$$
F_i = \Big|\sum_{j \in \mathrm{friends}(i)} \sigma_j\Big|,
\qquad
E_i = \Big|\sum_{j \in \mathrm{enemies}(i)} \sigma_j\Big|,
$$

whose difference $y_i = F_i - E_i$ is the *alignment asymmetry* ([`alignment_asymmetry()`](#pycoop.observables.alignment_asymmetry)).
A positive $y_i$ says player $i$'s friends outweigh their enemies, leaving no local incentive to defect.

A state is *fraught* when three things hold at once: no player sits at zero, the crowd is split, and every $y_i \geq 0$.
That is a standoff, everyone locally content while the group has failed to coordinate, and exactly a non-Pareto-optimal Nash equilibrium (Tabor, Smith, and Dankowicz, 2024; Conrad and Tabor, 2024).
Such states are tense but stable: fraught, not frustrated.
Fraughtness needs at least four players.
On the paw it occupies a sliver of volume $1/384$ of the hypercube, on the four-path a healthier $1/32$, both reproduced by Monte Carlo with [`is_fraught()`](#pycoop.observables.is_fraught).

## Obdurateness

The other way to fail.
A state is *obdurate* when the crowd is split, no player is at zero, and some player has $y_i < 0$.
That player could improve their agreement by switching sides, but the dynamics won't let them, their agreement already above threshold and their window shut.
The jam is the work of holdouts, each enjoying high agreement bought by their neighbors' coordination while contributing nothing to it.
[`is_obdurate()`](#pycoop.observables.is_obdurate) tests the condition, and [`classify()`](#pycoop.observables.classify) sorts any state into `consensus`, `fraught`, `obdurate`, or `indeterminate`.

## Over a whole run

[`classify_trajectory()`](#pycoop.observables.classify_trajectory) labels every event state, and [`state_fractions()`](#pycoop.observables.state_fractions) weights those labels by the time spent in them:

In [ ]:
# Install pyCoop first when this notebook runs somewhere it is not installed,
# such as Google Colab. Where pyCoop is already installed this does nothing.
try:
    import pycoop
except ImportError:
    %pip install --quiet git+https://github.com/EternalTime/pyCoop.git

In [ ]:
import pycoop as pc
from pycoop import observables as obs

game = pc.Game(pc.erg_connected(10, 0.1, rng=0), u=0.5, l=0.2, T=0.1)
traj = game.run(rng=1)
print(obs.state_fractions(traj))

On the complete graph one more number is worth extracting.
While the window is open, flips only ever depolarize the crowd, and the mean opinion decays at a rate set by the fraction of players past the position threshold,

$$
\alpha = 1 - \frac{2l}{T + 2},
$$

a geometric occupancy fraction, which is why it holds still.
[`alpha_estimate()`](#pycoop.observables.alpha_estimate) measures it from a trajectory; on $K_{64}$ the estimate matches the formula to four digits.
Voila!